# Modelagem Supervisionada

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
    precision_score, 
    recall_score, 
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score
)
import plotly.graph_objects as go
import plotly.figure_factory as ff 


pd.set_option('display.max_columns', None)

In [9]:
#importando dados
df = pd.read_pickle('../data/curated/features.pkl')
df.head()

,IS_DELAYED,AIRPLANE_WAS_DELAYED,ORIGIN_AIRPORT_DELAY_MOMENTUM,DESTINATION_AIRPORT_DELAY_MOMENTUM,ORIGIN_AIRPORT_FLIGHTS_ON_THE_SAME_WINDOW,MONTH,SCHEDULED_DEPARTURE_HOUR,IS_WEEKEND,IS_HOLIDAY,HAUL_TYPE_SHORT,HAUL_TYPE_MEDIUM,HAUL_TYPE_LONG
0,1,-0.547400,-0.898713,-0.904391,-0.887294,-0.338099,-1.588269,1.716522,-0.161727,0,0,1
1,1,1.826816,-0.898713,0.719494,-1.046324,-1.521193,-0.749592,-0.582573,-0.161727,0,1,0
2,1,-0.547400,0.490011,0.351049,-0.887294,1.732315,-1.168931,-0.582573,-0.161727,1,0,0
3,0,-0.547400,0.490011,0.507980,-0.569234,0.253448,0.508425,-0.582573,-0.161727,0,1,0
4,0,-0.547400,-0.898713,-0.464987,-0.410204,0.844994,-2.846286,-0.582573,-0.161727,0,0,1


Treinamento

In [10]:
X = df.drop(columns=['IS_DELAYED'])
y = df['IS_DELAYED']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Distribuição das classes:\n{y.value_counts(normalize=True)}')
print(f'Treino: {X_train.shape}, Teste: {X_test.shape}')

Distribuição das classes:
IS_DELAYED
1    0.5
0    0.5
Name: proportion, dtype: float64
Treino: (1543796, 11), Teste: (385950, 11)


In [ ]:
def evaluate_model(model, X_test, y_test):
    '''Função auxiliar para métricas.'''
    y_pred = model.predict(X_test)
    return {
        'Acc Test': accuracy_score(y_test, y_pred),
        'F1 Test': f1_score(y_test, y_pred, average='weighted'),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall': recall_score(y_test, y_pred, average='weighted')
    }

def run_model_selection(X_train, y_train, X_test, y_test):
    '''
    Testa modelos pré-definidos e retorna o nome do melhor.
    Recebe apenas os dois pares de parâmetros (X, y) de treino e teste.
    '''
    models = {
        'XGBoost': XGBClassifier(random_state=42, tree_method='hist'),
        'RandomForest': RandomForestClassifier(random_state=42)
    }
    
    params = {
        'XGBoost': {'n_estimators': [100, 200], 'learning_rate': [0.1, 0.05]},
        'RandomForest': {'n_estimators': [100, 200], 'max_depth': [10, 20]}
    }

    print('Iniciando seleção de modelos...')
    summary = []

    for name in models.keys():
        print(f'Avaliando: {name}...')
        search = RandomizedSearchCV(
            estimator=models[name],
            param_distributions=params[name],
            n_iter=2,
            scoring='f1_weighted',
            cv=3,
            random_state=42,
            n_jobs=-1
        )
        search.fit(X_train, y_train)
        
        metrics = evaluate_model(search.best_estimator_, X_test, y_test)
        metrics['Model'] = name
        metrics['Best CV Score'] = search.best_score_
        summary.append(metrics)

    df_results = pd.DataFrame(summary).sort_values(by='F1 Test', ascending=False)
    best_model = df_results.iloc[0]['Model']
    
    print('\n--- Ranking de Modelos ---')
    print(df_results[['Model', 'F1 Test', 'Acc Test']])
    
    return best_model, df_results

In [12]:
best_model, _ = run_model_selection(X_train, y_train, X_test, y_test)

Iniciando seleção rápida de modelos...
Avaliando: XGBoost...
Avaliando: RandomForest...

--- Ranking de Modelos ---
          Model   F1 Test  Acc Test
0       XGBoost  0.745458  0.747952
1  RandomForest  0.743320  0.745752


In [13]:
def run_fine_tuning(X_train, y_train, X_test, y_test, model):
    '''
    Aplica busca exaustiva no modelo vencedor.
    '''
    print(f'\nIniciando Fine-Tuning exaustivo para: {model}')
    
    if model == 'XGBoost':
        model = XGBClassifier(random_state=42, tree_method='hist')
        params = {
            'n_estimators': [100, 300, 500],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 6, 9],
            'subsample': [0.8, 1.0]
        }
    else:
        model = RandomForestClassifier(random_state=42)
        params = {
            'n_estimators': [100, 300, 500],
            'max_depth': [10, 20, None],
            'min_samples_split': [2, 5, 10]
        }

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=10,
        scoring='f1_weighted',
        cv=5,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    
    search.fit(X_train, y_train)
    model = search.best_estimator_
    
    y_pred = model.predict(X_test)
    print('\n' + '='*40)
    print(f'Modelo: {model}')
    print('='*40)
    print(f'Melhores parâmetros: {search.best_params_}')
    print('\nRelatório de classificação:')
    print(classification_report(y_test, y_pred))
    
    return model

In [14]:
model = run_fine_tuning(X_train, y_train, X_test, y_test, best_model)


Iniciando Fine-Tuning exaustivo para: XGBoost
Fitting 5 folds for each of 10 candidates, totalling 50 fits


/home/platero/postech-ml-engineering-fase-3-tech-challenge-api/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Modelo: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=9,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)
Melhores parâmetros: {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.05}

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.71      0.85      0.77    192

In [15]:
def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    x = ['Previsto Pontual', 'Previsto Atrasado']
    y = ['Real Pontual', 'Real Atrasado']
    
    fig = ff.create_annotated_heatmap(
        z=cm, 
        x=x, 
        y=y, 
        annotation_text=cm, 
        colorscale='Blues'
    )
    fig.update_layout(
        title='Matriz de Confusão',
        xaxis_title='Predição',
        yaxis_title='Realidade',
        template='plotly_white'
    )
    fig.show()


y_pred = model.predict(X_test)
plot_confusion_matrix(y_test, y_pred)

In [16]:
def plot_roc_curve(y_true, y_probs):
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    auc_score = roc_auc_score(y_true, y_probs)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
            x=fpr, 
            y=tpr,
            mode='lines',
            name=f'AUC = {auc_score:.4f}',
            line=dict(color='darkblue', width=3)
        )
    )
    fig.add_trace(go.Scatter(
            x=[0, 1], 
            y=[0, 1],
            mode='lines',
            name='Predição Aleatória',
            line=dict(color='red', dash='dash')
        )
    )
    fig.update_layout(
        title='Curva ROC',
        xaxis_title='Taxa de Falso Positivo (1 - Especificidade)',
        yaxis_title='Taxa de Verdadeiro Positivo (Sensibilidade)',
        height=600,
        template='plotly_white'
    )
    fig.show()


y_probs = model.predict_proba(X_test)[:, 1]
plot_roc_curve(y_test, y_probs)

**Que características aumentam a chance de atraso em um voo?**

In [17]:
def plot_feature_importances(df):
    fig = go.Figure(
        data=[
            go.Bar(
                x=df['Importância'],
                y=df['Variável'],
                orientation='h',
                marker=dict(
                    color=df['Importância'],
                    colorscale='Blues',
                    showscale=False
                )
            )
        ]
    )
    fig.update_layout(
        title={
            'text': 'Importância das Variáveis Explanatórias',
            'y': 0.95,
            'x': 0.05,
            'xanchor': 'left',
            'yanchor': 'top'
        },
        xaxis_title='Importância',
        yaxis_title='Variável',
        yaxis=dict(autorange='reversed'),
        margin=dict(l=150, r=30, t=60, b=50),
        template='plotly_white',
        height=500
    )
    fig.show()


df_importances = pd.DataFrame({
    'Variável': X.columns,
    'Importância': model.feature_importances_
}).sort_values(by='Importância', ascending=False)

plot_feature_importances(df_importances)